In [ ]:
import numpy as np

from cmapss.config import load_config
from cmapss.models import sequences_to_tabular, train_ridge, train_random_forest
from cmapss.evaluate import evaluate, naive_baseline_rmse
from cmapss.viz import plot_prediction_vs_actual

# TODO(다음 단계): XGBoost, LSTM/RNN, MC Dropout LSTM(+coverage), Attention LSTM

cfg = load_config()
processed_dir = cfg["paths"]["processed_dir"]
SEED = cfg["preprocess"]["seed"]

data = {name: np.load(processed_dir / f"{name}.npy")
        for name in ["X_train", "y_train", "X_val", "y_val", "X_test", "y_test"]}
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

print("X_train", X_train.shape, "X_val", X_val.shape, "X_test", X_test.shape)

In [ ]:
Xtr_tab, Xval_tab, Xte_tab = (sequences_to_tabular(X) for X in (X_train, X_val, X_test))

print(f"Naive baseline (train 평균) — val RMSE={naive_baseline_rmse(y_train, y_val):.2f}, "
      f"test RMSE={naive_baseline_rmse(y_train, y_test):.2f}")

models = {
    "Ridge": train_ridge(Xtr_tab, y_train),
    "RandomForest": train_random_forest(Xtr_tab, y_train, seed=SEED),
}
results = []
for name, model in models.items():
    results.append({**evaluate(y_val, model.predict(Xval_tab), f"{name} / val"), "split": "val"})
    results.append({**evaluate(y_test, model.predict(Xte_tab), f"{name} / test"), "split": "test"})

In [ ]:
plot_prediction_vs_actual(y_test, models["RandomForest"].predict(Xte_tab),
                          title="RandomForest — test (FD001)");